<a href="https://colab.research.google.com/github/Magdal04/Natural-Language-Processing/blob/main/LAB02_Sistem_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lucrare de laborator Nr. 2
## Sistem automat de Întrebare-Răspuns pe baza unui articol Wikipedia

> Sursa textului: pagina în limba română Wikipedia despre Taylor Swift.

### Obiective
- extragerea și curățarea unui text mai lung de 5000 de cuvinte
- preprocesare NLP cu spaCy și NLTK
- formularea a 20 de întrebări simple în limba română
- identificarea celor mai relevante fragmente și generarea răspunsurilor

> Notă: notebookul salvează un snapshot local al textului extras pentru reproductibilitate.

In [ ]:
from pathlib import Path
import re
import json
import requests
import urllib3
import nltk
import spacy
from requests.exceptions import SSLError
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer

try:
    import pandas as pd
except ImportError:
    pd = None

nltk.download("stopwords", quiet=True)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

ARTICLE_TITLE = "Taylor_Swift"
ARTICLE_URL = "https://ro.wikipedia.org/wiki/Taylor_Swift"
SNAPSHOT_PATH = DATA_DIR / "taylor_swift_wikipedia_ro.txt"
QUESTION_SNAPSHOT_PATH = DATA_DIR / "qa_results_taylor_swift.json"
REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

tokenizer = RegexpTokenizer(r"\w+")
ro_stopwords = set(stopwords.words("romanian"))
question_words = {
    "cine", "ce", "unde", "când", "cand", "cum", "care", "cât", "câți",
    "câte", "de", "la", "în", "din", "pe", "este", "a", "s", "sa"
}

def load_romanian_pipeline():
    model_candidates = ["ro_core_news_lg", "ro_core_news_md", "ro_core_news_sm"]
    for model_name in model_candidates:
        try:
            pipeline = spacy.load(model_name)
            return pipeline, model_name, True, True
        except OSError:
            continue

    pipeline = spacy.blank("ro")
    if "sentencizer" not in pipeline.pipe_names:
        pipeline.add_pipe("sentencizer")
    return pipeline, "spacy.blank('ro')", False, False

def request_with_ssl_fallback(url: str, **kwargs):
    request_kwargs = {"headers": REQUEST_HEADERS, **kwargs}
    try:
        return requests.get(url, **request_kwargs)
    except SSLError:
        return requests.get(url, verify=False, **request_kwargs)

def fetch_wikipedia_plaintext(title: str) -> str:
    api_url = "https://ro.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "titles": title,
        "explaintext": 1,
        "redirects": 1,
        "format": "json",
        "formatversion": 2,
    }
    response = request_with_ssl_fallback(api_url, params=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    pages = payload.get("query", {}).get("pages", [])
    if not pages or "extract" not in pages[0]:
        raise ValueError("Nu am putut extrage textul articolului Wikipedia.")
    return pages[0]["extract"]

def clean_wikipedia_text(raw_text: str) -> str:
    stop_sections = {
        "Discografie",
        "Turnee",
        "Filmografie",
        "Referințe",
        "Legături externe",
        "Note",
        "Bibliografie",
    }

    cleaned_lines = []
    skip_rest = False

    for line in raw_text.splitlines():
        stripped = line.strip()
        if not stripped:
            cleaned_lines.append("")
            continue

        if stripped in stop_sections:
            skip_rest = True
        if skip_rest:
            continue

        if stripped.startswith("^") or stripped.startswith("[modificare") or stripped.startswith("Salt "):
            continue

        line = re.sub(r"\[\d+\]", "", stripped)
        line = re.sub(r"\s+", " ", line).strip()
        if line:
            cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    return text.strip()

nlp, loaded_model_name, pos_available, ner_available = load_romanian_pipeline()

raw_text = fetch_wikipedia_plaintext(ARTICLE_TITLE)
clean_text = clean_wikipedia_text(raw_text)
SNAPSHOT_PATH.write_text(clean_text, encoding="utf-8")

word_count = len(clean_text.split())
doc = nlp(clean_text)
sentences = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

print(f"Model spaCy folosit: {loaded_model_name}")
print(f"POS disponibil: {pos_available}")
print(f"NER disponibil: {ner_available}")
print(f"URL sursă: {ARTICLE_URL}")
print(f"Fișier snapshot: {SNAPSHOT_PATH}")
print(f"Număr total de cuvinte: {word_count}")
print(f"Număr de propoziții: {len(sentences)}")
print()
print("Primele 3 propoziții din corpus:")
for index, sentence in enumerate(sentences[:3], start=1):
    print(f"{index}. {sentence}")

if not pos_available:
    print()
    print("Atenție: modelul român preantrenat spaCy nu este instalat în acest kernel.")
    print("Notebookul rulează cu un fallback bazat pe spacy.blank('ro'), ceea ce păstrează segmentarea în propoziții, dar reduce calitatea POS și NER.")

if word_count <= 5000:
    raise ValueError("Textul extras nu depășește 5000 de cuvinte. Ajustează filtrarea secțiunilor.")

Model spaCy folosit: ro_core_news_sm
POS disponibil: True
NER disponibil: True
URL sursă: https://ro.wikipedia.org/wiki/Taylor_Swift
Fișier snapshot: d:\Gorincioi_Igor\Python\Master\NLP\data\taylor_swift_wikipedia_ro.txt
Număr total de cuvinte: 6613
Număr de propoziții: 256

Primele 3 propoziții din corpus:
1. Taylor Alison Swift (n. 13 decembrie 1989, West Reading⁠(d), Pennsylvania, SUA) este o cântăreață și compozitoare americană.
2. Recunoscută pentru versurile sale autobiografice, versatilitatea și reinvențiile artistice, și impactul cultural, Swift este o figură proeminentă a secolului XXI, care a generat o adevărată influență culturală, numită efectul Taylor Swift.
3. Vânzând peste 200 de milioane de còpii la nivel global, Swift este unul dintre cei mai bine vânduți artiști din istorie, cea cu cele mai mari încasări din turnee din istorie, și primul miliardar cu muzica ca principală sursă de venituri.


## Întrebări și strategie QA

> Mai jos definim 20 de întrebări factuale în limba română și implementăm un sistem extractiv simplu:
- detectăm tipul întrebării
- extragem cuvinte-cheie și leme
- ordonăm propozițiile după relevanță
- formulăm un răspuns scurt sau folosim propoziția cea mai potrivită

In [ ]:
questions = [
    "Când s-a născut Taylor Swift?",
    "Unde s-a născut Taylor Swift?",
    "Cum se numește fratele mai mic al lui Taylor Swift?",
    "La ce vârstă a început Taylor Swift să se intereseze de teatrul muzical?",
    "În ce oraș a mers Taylor Swift pentru a-și lansa cariera muzicală?",
    "Cu ce casă de discuri a semnat Taylor Swift în 2005?",
    "În ce an a fost lansat albumul de debut Taylor Swift?",
    "Cum se numește al doilea album al artistei?",
    "În ce an a fost lansat albumul Fearless?",
    "Ce album a marcat tranziția lui Taylor Swift spre pop?",
    "În ce oraș s-a mutat Taylor Swift în 2014?",
    "Cum se numește turneul care a devenit cel mai de succes turneu din istorie?",
    "În ce an a început The Eras Tour?",
    "Ce album a fost lansat pe 19 aprilie 2024?",
    "Ce album a fost lansat pe 21 octombrie 2022?",
    "Ce piesă a ocupat poziția #1 în săptămâna de debut a albumului Midnights?",
    "În ce an a semnat Taylor Swift cu Republic Records?",
    "Ce documentar despre Taylor Swift a fost lansat pe Netflix în 2020?",
    "Ce titlu i-a acordat revista Time în 2023?",
    "Câte premii Grammy are Taylor Swift?"
]

question_type_aliases = {
    "cine": "cine",
    "ce": "ce",
    "unde": "unde",
    "când": "cand",
    "cand": "cand",
    "cum": "cum",
    "care": "care",
    "cât": "cat",
    "câți": "cat",
    "câte": "cat",
}

answer_entity_map = {
    "unde": {"LOC", "GPE", "FAC"},
    "cand": {"DATE", "TIME"},
    "cine": {"PER", "PERSON", "ORG"},
}

global_noise_words = {
    "taylor", "swift", "artista", "artistei", "artistă", "albumul",
    "album", "piesa", "piesa", "turneul", "în", "ce", "a", "s",
}

def detect_question_type(question: str) -> str:
    first_word = tokenizer.tokenize(question.lower())[0]
    return question_type_aliases.get(first_word, "altul")

def normalize_tokens_with_nltk(text: str) -> list[str]:
    tokens = tokenizer.tokenize(text.lower())
    return [token for token in tokens if token.isalnum()]

def extract_focus_terms(question: str) -> set[str]:
    quoted_terms = {match.lower() for match in re.findall(r"[„\"]([^\"”]+)[\"”]", question)}
    title_terms = {
        token.lower()
        for token in re.findall(r"\b[A-ZĂÂÎȘȚ][A-Za-zĂÂÎȘȚăâîșț'\-]+\b", question)
        if token.lower() not in global_noise_words
    }
    numeric_terms = set(re.findall(r"\b\d{4}\b", question))
    return quoted_terms | title_terms | numeric_terms

def extract_question_profile(question: str) -> dict:
    doc_q = nlp(question)
    raw_tokens = normalize_tokens_with_nltk(question)
    lemmas = []
    keywords = []

    for token in doc_q:
        lemma = token.lemma_.lower().strip() if token.lemma_ else token.text.lower().strip()
        text = token.text.lower().strip()
        if token.is_punct or token.is_space or not text.replace("-", "").isalnum():
            continue
        if text in ro_stopwords or text in question_words or text in global_noise_words:
            continue
        keywords.append(text)
        lemmas.append(lemma if lemma else text)

    if not keywords:
        keywords = [
            token for token in raw_tokens
            if token not in ro_stopwords and token not in question_words and token not in global_noise_words
        ]
        lemmas = keywords.copy()

    return {
        "question": question,
        "type": detect_question_type(question),
        "keywords": set(keywords),
        "lemmas": set(lemmas),
        "focus_terms": extract_focus_terms(question),
        "doc": doc_q,
    }

sentence_profiles = []
for sentence in sentences:
    clean_sentence = re.sub(r"=+", " ", sentence).strip()
    sent_doc = nlp(clean_sentence)
    sent_tokens = {token.text.lower() for token in sent_doc if token.text.strip() and token.text.replace("-", "").isalnum()}
    sent_lemmas = {
        (token.lemma_.lower() if token.lemma_ else token.text.lower())
        for token in sent_doc
        if token.text.strip() and token.text.replace("-", "").isalnum()
    }
    sentence_profiles.append(
        {
            "text": clean_sentence,
            "doc": sent_doc,
            "tokens": sent_tokens,
            "lemmas": sent_lemmas,
            "entities": [(ent.text, ent.label_) for ent in sent_doc.ents],
            "lower_text": clean_sentence.lower(),
        }
    )

def score_sentence(question_profile: dict, sentence_profile: dict) -> float:
    keyword_overlap = len(question_profile["keywords"] & sentence_profile["tokens"])
    lemma_overlap = len(question_profile["lemmas"] & sentence_profile["lemmas"])
    focus_matches = sum(
        1 for focus_term in question_profile["focus_terms"]
        if focus_term in sentence_profile["lower_text"]
    )
    focus_misses = sum(
        1 for focus_term in question_profile["focus_terms"]
        if focus_term not in sentence_profile["lower_text"]
    )

    entity_bonus = 0.0
    expected_labels = answer_entity_map.get(question_profile["type"], set())
    if expected_labels and any(label in expected_labels for _, label in sentence_profile["entities"]):
        entity_bonus = 1.5

    temporal_bonus = 0.0
    if question_profile["type"] == "cand" and re.search(r"\b(1\d{3}|20\d{2})\b", sentence_profile["text"]):
        temporal_bonus = 1.0

    taylor_version_penalty = 0.0
    if "taylor's version" in sentence_profile["lower_text"] and "taylor's version" not in question_profile["question"].lower():
        taylor_version_penalty = 2.0

    return (
        keyword_overlap * 1.5
        + lemma_overlap * 2.0
        + focus_matches * 3.0
        + entity_bonus
        + temporal_bonus
        - focus_misses * 0.75
        - taylor_version_penalty
    )

def get_top_sentences(question_profile: dict, top_k: int = 3) -> list[dict]:
    scored = []
    for sentence_profile in sentence_profiles:
        score = score_sentence(question_profile, sentence_profile)
        if score > 0:
            scored.append({**sentence_profile, "score": round(score, 2)})

    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:top_k]

def extract_short_answer(question_profile: dict, sentence_profile: dict) -> str:
    sentence = sentence_profile["text"]
    doc_sent = sentence_profile["doc"]
    question_text = question_profile["question"].lower()
    question_type = question_profile["type"]

    if "fratele mai mic" in question_text:
        match = re.search(r"frate mai\s+mic,\s+([A-ZĂÂÎȘȚ][^,.]+)", sentence)
        if match:
            return match.group(1).strip()

    if "al doilea album" in question_text:
        match = re.search(r"Al doilea album al artistei,\s*([^,]+)", sentence, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()

    if "ce documentar" in question_text:
        match = re.search(r"documentarul\s+[„\"]([^\"”]+)[\"”]", sentence, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()

    if "ce titlu" in question_text and "time" in question_text:
        match = re.search(r"Time\s+a numit-o\s+([A-Za-z ]+)", sentence)
        if match:
            return match.group(1).strip()

    if question_type == "unde":
        for ent_text, ent_label in sentence_profile["entities"]:
            if ent_label in answer_entity_map["unde"]:
                return ent_text
        match = re.search(r"\b(în|la|din)\s+([A-ZĂÂÎȘȚ][^,.]+)", sentence)
        if match:
            return f"{match.group(1)} {match.group(2).strip()}"

    if question_type == "cand":
        for ent_text, ent_label in sentence_profile["entities"]:
            if ent_label in answer_entity_map["cand"]:
                return ent_text
        if re.search(r"\b\d{1,2}\s+[a-zăâîșț]+\s+\d{4}\b", sentence.lower()):
            return re.search(r"\b\d{1,2}\s+[a-zăâîșț]+\s+\d{4}\b", sentence.lower()).group(0)
        match = re.search(r"\b(1\d{3}|20\d{2})\b", sentence)
        if match:
            return match.group(1)

    if question_type == "cine":
        for ent_text, ent_label in sentence_profile["entities"]:
            if ent_label in answer_entity_map["cine"]:
                return ent_text
        for token in doc_sent:
            if token.dep_ in {"nsubj", "nsubj:pass"}:
                subtree = " ".join(tok.text for tok in token.subtree)
                return subtree

    return sentence

def build_final_answer(question_profile: dict, top_candidates: list[dict]) -> tuple[str, str, float]:
    if not top_candidates:
        return "Nu a fost găsit un răspuns relevant în text.", "", 0.0

    best_candidate = top_candidates[0]
    short_answer = extract_short_answer(question_profile, best_candidate)

    if short_answer and short_answer != best_candidate["text"]:
        final_answer = f"Răspuns scurt: {short_answer}."
    else:
        final_answer = best_candidate["text"]

    return final_answer, best_candidate["text"], best_candidate["score"]

print(f"Număr de întrebări definite: {len(questions)}")
for preview_question in questions[:5]:
    print("-", preview_question)

Număr de întrebări definite: 20
- Când s-a născut Taylor Swift?
- Unde s-a născut Taylor Swift?
- Cum se numește fratele mai mic al lui Taylor Swift?
- La ce vârstă a început Taylor Swift să se intereseze de teatrul muzical?
- În ce oraș a mers Taylor Swift pentru a-și lansa cariera muzicală?


In [ ]:
results = []

for question in questions:
    question_profile = extract_question_profile(question)
    top_candidates = get_top_sentences(question_profile, top_k=3)
    final_answer, best_fragment, best_score = build_final_answer(question_profile, top_candidates)

    results.append(
        {
            "Întrebare": question,
            "Tip": question_profile["type"],
            "Scor": best_score,
            "Fragment relevant": best_fragment,
            "Răspuns final": final_answer,
        }
    )

QUESTION_SNAPSHOT_PATH.write_text(
    json.dumps(results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"Rezultatele au fost salvate în: {QUESTION_SNAPSHOT_PATH}")
print()

if pd is not None:
    results_df = pd.DataFrame(results)
    display(results_df)
else:
    for item in results:
        print(json.dumps(item, ensure_ascii=False, indent=2))

Rezultatele au fost salvate în: d:\Gorincioi_Igor\Python\Master\NLP\data\qa_results_taylor_swift.json

{
  "Întrebare": "Când s-a născut Taylor Swift?",
  "Tip": "cand",
  "Scor": 7.5,
  "Fragment relevant": "La sfârșitul lui 2019, Billboard și-a sărbătorit a 125-a aniversare, publicând „Topul Celor Mai Mari Artiști din Istorie”, unde Swift s-a clasat pe locul #8, între Michael Jackson și Stevie Wonder, fiind singura artistă din top 10 care a debutat în acest secol, dar și singura din top 10 cu mai puțin de 30 de ani de activitate, cu doar 13 de ani de carieră la acea dată.",
  "Răspuns final": "Răspuns scurt: 2019."
}
{
  "Întrebare": "Unde s-a născut Taylor Swift?",
  "Tip": "unde",
  "Scor": 8.0,
  "Fragment relevant": "La sfârșitul lui 2019, Billboard și-a sărbătorit a 125-a aniversare, publicând „Topul Celor Mai Mari Artiști din Istorie”, unde Swift s-a clasat pe locul #8, între Michael Jackson și Stevie Wonder, fiind singura artistă din top 10 care a debutat în acest secol, dar ș

## Observații finale

- sursa este extrasă automat din Wikipedia și salvată local pentru reproductibilitate
- răspunsurile sunt generate printr-un QA extractiv bazat pe ranking de propoziții
- pentru întrebările de tip unde, când și cine se încearcă întâi extragerea unui răspuns scurt
- sistemul este potrivit pentru întrebări factuale; pentru întrebări interpretative va avea limitări